# Training Notebook

This notebook trains the CNN Stock Market Prediction model.

## Contents
1. Load and prepare data
2. Initialize model and trainer
3. Train the model
4. Visualize training progress
5. Evaluate on test set
6. Save final model

In [ ]:
# Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.utils.config import print_config, DEVICE, BATCH_SIZE, NUM_EPOCHS

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')
print_config()

## 1. Load and Prepare Data

We'll load data for a few stocks and create training, validation, and test sets.

In [ ]:
from src.data.fetcher import fetch_stock_data, fetch_multiple_stocks, fetch_sp500_tickers
from src.data.preprocessor import create_sliding_windows, train_val_test_split
from src.data.dataset import create_dataloaders

# Configuration
WINDOW_SIZE = 256
HORIZON = 5  # 5-day prediction horizon
START_DATE = '2010-01-01'
END_DATE = '2024-01-01'

# Select a subset of stocks for faster training (expand to full S&P 500 later)
sample_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM', 'JNJ', 'V', 'PG']
print(f'Training on {len(sample_tickers)} stocks: {sample_tickers}')

In [ ]:
# Fetch data for all selected stocks
print('Fetching stock data...')
stock_data = fetch_multiple_stocks(sample_tickers, START_DATE, END_DATE)

print(f'\nSuccessfully fetched {len(stock_data)} stocks')
for ticker, df in stock_data.items():
    print(f'  {ticker}: {len(df)} days')

In [ ]:
# Create sliding windows for all stocks
print('Creating sliding windows...')

all_X = []
all_y = []

for ticker, df in stock_data.items():
    X, y = create_sliding_windows(df, window_size=WINDOW_SIZE, horizon=HORIZON)
    all_X.append(X)
    all_y.append(y)
    print(f'  {ticker}: {len(X)} windows')

# Combine all data
X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print(f'\nTotal: {len(X_all)} windows')
print(f'X shape: {X_all.shape}')
print(f'Label distribution: {y_all.mean():.2%} bullish')

In [ ]:
# Split into train/val/test (chronologically)
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(
    X_all, y_all, val_ratio=0.15, test_ratio=0.15
)

print(f'Train: {len(X_train)} samples ({y_train.mean():.2%} bullish)')
print(f'Validation: {len(X_val)} samples ({y_val.mean():.2%} bullish)')
print(f'Test: {len(X_test)} samples ({y_test.mean():.2%} bullish)')

In [ ]:
# Create DataLoaders
train_loader, val_loader = create_dataloaders(
    X_train, y_train, X_val, y_val, batch_size=BATCH_SIZE
)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')

# Quick sanity check
X_batch, y_batch = next(iter(train_loader))
print(f'\nBatch shape: {X_batch.shape}')

## 2. Initialize Model and Trainer

In [ ]:
from src.models.cnn import StockCNN
from src.training.trainer import Trainer
from src.utils.config import MODELS_DIR

# Create model
model = StockCNN()
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoint_dir=MODELS_DIR,
)

print('Trainer initialized successfully')
print(f'Checkpoint directory: {MODELS_DIR}')

## 3. Train the Model

Run the training loop. On CPU, expect each epoch to take several minutes.

In [ ]:
# Training parameters
# For a quick test, use fewer epochs
# For full training, increase to 50-100 epochs
TRAINING_EPOCHS = 10  # Adjust as needed
EARLY_STOPPING_PATIENCE = 5

print(f'Starting training for up to {TRAINING_EPOCHS} epochs...')
print(f'Early stopping patience: {EARLY_STOPPING_PATIENCE} epochs')
print('-' * 50)

In [ ]:
# Run training
metrics = trainer.train(
    num_epochs=TRAINING_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    checkpoint_interval=5,
)

## 4. Visualize Training Progress

In [ ]:
from src.visualization.plots import plot_training_summary, plot_loss_curves, plot_accuracy_curves

# Plot combined training summary
fig = plot_training_summary(metrics, title='Training Progress')
plt.show()

In [ ]:
# Detailed loss curves
train_loss = metrics.get_history('train_loss')
val_loss = metrics.get_history('val_loss')

fig = plot_loss_curves(train_loss, val_loss, title='Loss During Training')
plt.show()

In [ ]:
# Detailed accuracy curves
train_acc = metrics.get_history('train_accuracy')
val_acc = metrics.get_history('val_accuracy')

fig = plot_accuracy_curves(train_acc, val_acc, title='Accuracy During Training')
plt.show()

In [ ]:
# Print best results
best = metrics.get_best_metrics()
print('Best Results:')
print(f'  Validation Loss: {best["best_val_loss"]:.4f}')
print(f'  Validation Accuracy: {best["best_val_accuracy"]:.2%}')
print(f'  Best Epoch: {best["best_epoch"] + 1}')

## 5. Evaluate on Test Set

In [ ]:
from src.data.dataset import StockDataset
from torch.utils.data import DataLoader
from src.training.metrics import compute_confusion_matrix, compute_precision_recall_f1
from src.visualization.plots import plot_confusion_matrix, plot_prediction_distribution

# Create test DataLoader
test_dataset = StockDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Test set: {len(test_dataset)} samples, {len(test_loader)} batches')

In [ ]:
# Load best model
from src.training.trainer import load_model

best_model_path = MODELS_DIR / 'best_model.pt'
best_model = load_model(best_model_path)
print(f'Loaded best model from {best_model_path}')

In [ ]:
# Evaluate on test set
best_model.eval()

all_outputs = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        outputs = best_model(X_batch)
        all_outputs.append(outputs.cpu())
        all_targets.append(y_batch)

all_outputs = torch.cat(all_outputs, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Calculate metrics
predictions = all_outputs.argmax(dim=1)
test_accuracy = (predictions == all_targets).float().mean().item()

print(f'Test Accuracy: {test_accuracy:.2%}')

In [ ]:
# Precision, Recall, F1 for bullish class
metrics_dict = compute_precision_recall_f1(all_outputs, all_targets, class_idx=1)

print('Metrics for Bullish (BUY) Predictions:')
print(f'  Precision: {metrics_dict["precision"]:.2%}')
print(f'  Recall: {metrics_dict["recall"]:.2%}')
print(f'  F1 Score: {metrics_dict["f1"]:.2%}')

In [ ]:
# Confusion matrix
cm = compute_confusion_matrix(all_outputs, all_targets)

fig = plot_confusion_matrix(cm, title='Test Set Confusion Matrix')
plt.show()

In [ ]:
# Prediction confidence distribution
fig = plot_prediction_distribution(
    all_outputs.numpy(),
    all_targets.numpy(),
    title='Test Set Prediction Confidence'
)
plt.show()

## 6. Save Final Model

In [ ]:
# The best model has already been saved during training
# Let's also save training history

import json

history_path = MODELS_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(metrics.to_dict(), f, indent=2)

print(f'Training history saved to {history_path}')

# List saved models
print('\nSaved models:')
for model_file in MODELS_DIR.glob('*.pt'):
    print(f'  {model_file.name}')

## Summary

Training complete! The model has been trained and evaluated:

- Best model saved to `models/best_model.pt`
- Training history saved to `models/training_history.json`
- Test accuracy and other metrics computed

Next steps:
- Use the prediction notebook (`03_prediction.ipynb`) to make predictions on new data
- Try training with more stocks for better generalization
- Experiment with different horizons (T+5, T+30)